**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**01 | HARTH Data Foundation: Source Signals, Windowing and Statistical Features**

![Consolidated HARTH stage 01 workflow](../../images/01_consolidated_workflow.svg)

**Objective:** Examine a genuine HARTH v2.0 participant recording and produce one traceable Stage 01 workflow covering original signal inspection, time-safe label-pure windows and interpretable statistical features.

**Dataset:** Official NTNU AI Lab HARTH v2.0, pinned to the repository version. This integrated notebook uses `S023.csv`: two tri-axial accelerometers (lower back and thigh), a nominal sampling rate of 50 Hz and original acceleration values in **g**. The source differs from UCI HARTH v1.2.

**Notebook scope:** This notebook combines the former source-inspection, windowing and statistical-feature notebooks into one reusable stage. It reads the original recording once for the main processing path, audits the source schema and labels, visualises genuine signals, creates 100-sample windows with a 50-sample stride, extracts five statistical features per channel and saves the current Stage 01 outputs in `wearable_activity_recognition/output/`.

**Evaluation boundary:** S023 is used here only for the original exploratory demonstration. It was later among the two reserved HARTH assessment subjects and had already been inspected descriptively; therefore, later same-study held-out results must **not** be described as strictly blind or external. The multi-participant Development workflow belongs to a later stage.

**Reading order:** source file and schema → recorded activity labels → genuine signals → time-safe windows → retained activity support → example window → 30 statistical features → activity-level RMS exploration → saved outputs.

**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")

if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("wearable_activity_recognition")

# ----------------- 2. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 3. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure

plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11, legend_size=10,
    figure_dpi=100, figure_size=(9, 4),
)

# ----------------------- 4. Shared and Study Utilities -----------------------
from core.data_inspection import inspect_data, plot_signal
from core.signal_preprocessing import create_signal_windows
from core.feature_engineering import extract_signal_features
from core.output_download import download_outputs_from_colab
from harth_data import (
    ACTIVITY_NAMES, SENSOR_COLUMNS, ensure_harth_sample, load_harth_subject,
)

# ------------------- 5. Output Directory and Setup Summary -------------------
data_dir = root / "data" / "raw" / "harth_v2"
output_dir = root / "wearable_activity_recognition" / "output"
output_dir.mkdir(parents=True, exist_ok=True)

SUBJECT_ID = "S023"
SAMPLING_RATE = 50
WINDOW_SIZE = 100
STEP_SIZE = 50
FEATURE_METHODS = [
    "mean", "std", "rms", "peak_to_peak", "waveform_length",
]

print("Source:", f"HARTH v2.0 / {SUBJECT_ID}.csv")
print("Outputs:", output_dir)
print("Windowing:", WINDOW_SIZE, "samples, stride", STEP_SIZE)
print("Statistical features:", FEATURE_METHODS)


**02. Retrieve and Inspect the Original HARTH Recording**

The study adapter reuses `data/raw/harth_v2/S023.csv` if it is already present and nonempty. Otherwise it downloads the file from the official pinned source. A short raw-header inspection preserves the original column inventory without reading the full CSV twice. The full recording is then loaded once through the existing HARTH adapter, which parses timestamps and validates the six accelerometer channels and source labels.

The original recording has one annotated activity label per timestamp. This section reports the *observed* number of records and activities; no old-run record count is required to proceed.

In [ ]:
# ================= 02. Retrieve and Inspect Original Source ==================

# -------------------- 1. Source File and Original Schema ---------------------
csv_path = ensure_harth_sample(data_dir, subject=SUBJECT_ID)
source_header = pd.read_csv(csv_path, nrows=5)
print("Original file:", csv_path)
print("Original source columns:", source_header.columns.tolist())
display(source_header)

# ----------------------- 2. Validated Continuous Data ------------------------
data, channel_names = load_harth_subject(csv_path)
inspect_data(data, name=f"HARTH {SUBJECT_ID}: validated continuous recording")
print("Six accelerometer channels:", channel_names)
print("Sampling frequency (nominal):", SAMPLING_RATE, "Hz")
print("Original activity codes:", sorted(data["label"].unique().tolist()))

# ------------------------ 3. Source Coverage Summary -------------------------
observed_intervals = data["timestamp"].diff().dt.total_seconds().dropna()
source_summary = pd.DataFrame([{
    "subject_id": SUBJECT_ID,
    "source_file": csv_path.name,
    "recorded_samples": len(data),
    "source_columns": len(source_header.columns),
    "acceleration_channels": len(channel_names),
    "missing_values": int(data.isna().sum().sum()),
    "nominal_sampling_frequency_hz": SAMPLING_RATE,
    "median_observed_interval_s": float(observed_intervals.median()),
    "observed_activity_codes": data["label"].nunique(),
}])
display(source_summary)
source_summary.to_csv(output_dir / "01_source_inspection_summary.csv", index=False)


**03. Explore Original Activity Labels and Source Signals**

The recorded activity distribution is calculated from the original source samples, **before** segmentation. The lower-back X/Y/Z traces show genuine acceleration (g) without filtering, interpolation or scaling. Sample counts describe this participant's recording, not independent people.

Original codes are retained. A label without a documented description is shown as an unidentified source code rather than silently assigned to a different activity.

In [ ]:
# ============== 03. Original Activity and Acceleration Signals ===============

# --------------------- 1. Recorded Activity Distribution ---------------------
original_counts = data["label"].value_counts().sort_index()
activity_table = pd.DataFrame({
    "label": original_counts.index.astype(int),
    "activity": [
        ACTIVITY_NAMES.get(int(code), f"Unmapped source code {int(code)}")
        for code in original_counts.index
    ],
    "samples": original_counts.to_numpy(),
})
display(activity_table)
activity_table.to_csv(output_dir / "01_activity_distribution.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(activity_table["activity"], activity_table["samples"])
ax.set(
    title=f"HARTH {SUBJECT_ID}: annotated source samples",
    xlabel="Activity", ylabel="Recorded samples",
)
ax.tick_params(axis="x", labelrotation=45)
fig.tight_layout()
save_figure(fig, output_dir / "01_activity_distribution.png")
plt.show()

# ------------------------ 2. Genuine Unscaled Signal -------------------------
n_points = min(1000, len(data))
original_signal = data.iloc[:n_points][channel_names[:3]].to_numpy().T
fig = plot_signal(
    original_signal,
    title=f"HARTH {SUBJECT_ID}: lower-back acceleration (first {n_points} samples)",
    channel_names=channel_names[:3],
)
fig.axes[0].set_xlabel("Source sample index")
fig.axes[0].set_ylabel("Acceleration (g)")
save_figure(fig, output_dir / "01_acceleration_signals.png")
plt.show()


**04. Build Time-Safe, Label-Pure Signal Windows**

At 50 Hz, a 100-sample window represents two seconds. The 50-sample stride creates 50% overlap. The shared `create_signal_windows` function excludes a candidate if its labels differ or its timestamps contain a non-positive or excessive gap (above 1.5 times the expected sample period).

This operation runs **once in this notebook**. The returned tensor is `(windows, 6 channels, 100 samples)` and `window_info` retains original source-row indices and participant identity. No scaling or model fitting occurs.

In [ ]:
# ================== 04. Generate Label-Pure Signal Windows ===================

# ------------------- 1. Apply the Original Windowing Rules -------------------
X_windows, y_windows, window_info = create_signal_windows(
    signals=data[channel_names].to_numpy(),
    labels=data["label"].to_numpy(),
    window_size=WINDOW_SIZE,
    step_size=STEP_SIZE,
    timestamps=data["timestamp"],
    sampling_rate=SAMPLING_RATE,
    min_label_purity=1.0,
    subject_id=SUBJECT_ID,
)

# ----------------- 2. Verify Essential Array Correspondence ------------------
if not (len(X_windows) == len(y_windows) == len(window_info)):
    raise ValueError("Window arrays, labels and source metadata are misaligned.")
if not len(X_windows):
    raise ValueError("No time-safe, single-activity windows were retained.")

print("Signal windows:", X_windows.shape)
print("Window labels:", y_windows.shape)
print("Source metadata rows:", len(window_info))
display(window_info.head(8))

# ------------------------ 3. Save the Current Windows ------------------------
window_info.to_csv(output_dir / "01_window_metadata.csv", index=False)
np.savez_compressed(
    output_dir / "01_label_pure_windows.npz",
    X_windows=X_windows,
    y_windows=y_windows,
    channel_names=np.asarray(channel_names),
    window_size=np.int64(WINDOW_SIZE),
    step_size=np.int64(STEP_SIZE),
)
print("Saved source-aligned windows and metadata to:", output_dir)


**05. Compare Activity Coverage Before and After Windowing**

Count the source samples and the retained *two-second windows* for each activity. These are different units and should not be interpreted as a direct retention percentage. Transitions, discontinuities and incomplete windows can change the relative activity support after segmentation.

In [ ]:
# ============== 05. Compare Source and Window Activity Support ===============

# --------------------- 1. Source and Window Label Counts ---------------------
window_counts = pd.Series(y_windows, name="label").value_counts().sort_index()
window_distribution = pd.DataFrame({
    "label": original_counts.index.astype(int),
    "activity": [
        ACTIVITY_NAMES.get(int(code), f"Unmapped source code {int(code)}")
        for code in original_counts.index
    ],
    "source_samples": original_counts.to_numpy(),
    "retained_windows": [
        int(window_counts.get(code, 0)) for code in original_counts.index
    ],
})
display(window_distribution)
window_distribution.to_csv(
    output_dir / "01_window_activity_support.csv", index=False
)

# ------------------------- 2. Retained Window Figure -------------------------
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(window_distribution["activity"], window_distribution["retained_windows"])
ax.set(
    title=f"HARTH {SUBJECT_ID}: retained label-pure windows",
    xlabel="Activity", ylabel="Two-second windows",
)
ax.tick_params(axis="x", labelrotation=45)
fig.tight_layout()
save_figure(fig, output_dir / "01_retained_activity_distribution.png")
plt.show()


**06. Inspect a Genuine Two-Second Signal Window**

This figure uses the **first eligible window**, not a hand-picked favourable example. The accelerometer values remain in g. The original sample indices, source annotation and label purity are shown alongside the plot.

In [ ]:
# =================== 06. Visualise a Source-Aligned Window ===================

# ------------------------- 1. Original Window Trace --------------------------
example_index = 0
example_label = int(y_windows[example_index])
fig = plot_signal(
    X_windows[example_index, :3, :],
    title=(f"HARTH {SUBJECT_ID}: "
           f"{ACTIVITY_NAMES.get(example_label, str(example_label))} window"),
    channel_names=channel_names[:3],
)
fig.axes[0].set_xlabel("Sample index within the two-second window")
fig.axes[0].set_ylabel("Acceleration (g)")
save_figure(fig, output_dir / "01_example_window_signals.png")
plt.show()

# ----------------------- 2. Source Window Coordinates ------------------------
source_record = window_info.iloc[example_index]
print("Original source samples:", int(source_record["start_index"]), "to",
      int(source_record["end_index_exclusive"]) - 1)
print("Activity label:", example_label)
print("Label purity:", float(source_record["label_purity"]))


**07. Extract Thirty Interpretable Statistical Features**

Apply the same five predefined methods to each of the six acceleration channels: **mean**, **population standard deviation**, **root mean square (RMS)**, **peak-to-peak range** and **normalised waveform length** (mean absolute difference between adjacent samples).

This section reuses `X_windows` already produced in Section 04. It does **not** reload the source, rebuild a second window set or fit feature scaling. The resulting table contains 30 aligned per-window predictors and is exported together with the corresponding source-window metadata.

In [ ]:
# =========== 07. Extract Statistical Features from Current Windows ===========

# --------------------- 1. Deterministic Window Features ----------------------
feature_table = extract_signal_features(
    X_windows,
    channel_names=channel_names,
    methods=FEATURE_METHODS,
)

# --------------------- 2. Align Features and Source Rows ---------------------
if len(feature_table) != len(window_info):
    raise ValueError("Statistical feature rows do not align with source windows.")
if not np.isfinite(feature_table.to_numpy()).all():
    raise ValueError("The statistical feature table contains non-finite values.")

print("Feature matrix:", feature_table.shape)
print("Example feature names:", feature_table.columns[:10].tolist())
feature_overview = inspect_data(
    feature_table, name=f"HARTH {SUBJECT_ID}: statistical feature matrix"
)
display(feature_table.head())

# ------------------- 3. Save Current Aligned Feature Table -------------------
feature_export = pd.concat(
    [
        window_info.reset_index(drop=True).add_prefix("window_"),
        pd.DataFrame({"source_label": y_windows}),
        feature_table.reset_index(drop=True),
    ],
    axis=1,
)
feature_export.to_csv(output_dir / "01_statistical_features.csv", index=False)
print("Saved aligned feature table:",
      output_dir / "01_statistical_features.csv")


**08. Explore RMS Features Across Annotated Activities**

The scatter plot compares lower-back and thigh X-axis RMS, while the boxplot shows the distribution of lower-back X-axis RMS within each observed activity. These figures are descriptive feature analyses, **not classifier predictions**. Their differences reflect the single inspected participant and do not establish unseen-participant performance.

In [ ]:
# =============== 08. Visualise Activity-Dependent RMS Features ===============

# ------------------------ 1. Source-Labelled RMS Data ------------------------
rms_data = feature_table[["back_x__rms", "thigh_x__rms"]].copy()
rms_data["activity"] = [
    ACTIVITY_NAMES.get(int(label), f"Unmapped source code {int(label)}")
    for label in y_windows
]

# ---------------------------- 2. RMS Scatter Plot ----------------------------
fig, ax = plt.subplots(figsize=(9, 5))
for activity, subset in rms_data.groupby("activity", sort=True):
    ax.scatter(
        subset["back_x__rms"], subset["thigh_x__rms"],
        s=12, alpha=0.5, label=activity,
    )
ax.set(
    title=f"HARTH {SUBJECT_ID}: RMS movement features by activity",
    xlabel="Lower-back X RMS (g)", ylabel="Thigh X RMS (g)",
)
ax.legend(title="Activity", fontsize=8, markerscale=1.5)
fig.tight_layout()
save_figure(fig, output_dir / "01_rms_activity_scatter.png")
plt.show()

# -------------------- 3. Activity-Level RMS Measurements ---------------------
rms_summary = (
    rms_data.groupby("activity", sort=True)
    .agg(
        retained_windows=("back_x__rms", "size"),
        mean_back_x_rms=("back_x__rms", "mean"),
        mean_thigh_x_rms=("thigh_x__rms", "mean"),
    )
    .sort_values("retained_windows", ascending=False)
)
display(rms_summary)
rms_summary.reset_index().to_csv(
    output_dir / "01_rms_activity_summary.csv", index=False
)

# --------------------- 4. RMS Distributions by Activity ----------------------
activity_groups = list(rms_data.groupby("activity", sort=True))
fig, ax = plt.subplots(figsize=(10, 4))
ax.boxplot(
    [group["back_x__rms"].to_numpy() for _, group in activity_groups],
    tick_labels=[activity for activity, _ in activity_groups],
    showfliers=False,
)
ax.set(
    title=f"HARTH {SUBJECT_ID}: lower-back X RMS by activity",
    xlabel="Annotated activity", ylabel="RMS acceleration (g)",
)
ax.tick_params(axis="x", labelrotation=45)
fig.tight_layout()
save_figure(fig, output_dir / "01_rms_activity_boxplot.png")
plt.show()


**09. Save the Stage 01 Summary and Package Current Outputs**

All outputs from the former source-inspection, windowing and feature-engineering notebooks now belong to **Stage 01** and use the stable `01_` prefix in the study `output/` directory. A later stage can read these saved files directly; it does not need to inspect this notebook or compare any historical notebook version.

**Output note:** CSVs and PNGs are ordinary publication-ready research outputs after review. The compressed numeric NPZ is a local convenience artefact for later reuse of the current windows and may be kept outside Git tracking if required.

In [ ]:
# ================== 09. Summarise and Package Stage Outputs ==================

# -------------------- 1. Current Windowing Configuration ---------------------
window_summary = pd.DataFrame([{
    "subject_id": SUBJECT_ID,
    "sampling_rate_hz": SAMPLING_RATE,
    "window_size_samples": WINDOW_SIZE,
    "step_size_samples": STEP_SIZE,
    "overlap_fraction": 1 - STEP_SIZE / WINDOW_SIZE,
    "label_purity_threshold": 1.0,
    "source_rows": len(data),
    "retained_windows": len(X_windows),
    "signal_channels": len(channel_names),
    "statistical_features": feature_table.shape[1],
}])
window_summary.to_csv(output_dir / "01_windowing_summary.csv", index=False)
display(window_summary)
print("Saved Stage 01 summary:", output_dir / "01_windowing_summary.csv")
print("Stage 01 research files:", output_dir)

# ------------------------- 2. Manual Colab Download --------------------------
download_outputs_from_colab(
    folder="wearable_activity_recognition/output", stage="01"
)


**10. Scientific Interpretation and Next Research Boundary**

This integrated Stage 01 notebook preserves the substantive outputs of the former source-inspection, windowing and feature-engineering notebooks while removing repeated data loading, repeated window construction and administrative quality-gate logic. It examines original, unscaled acceleration signals in **g** and applies source-safe two-second segmentation. Each retained window uses a single annotated activity and retains its participant and source-sample coordinates. The same windows support five deterministic features for each of six acceleration channels, producing a 30-column feature matrix.

**Interpretation:** Source-sample counts and window counts describe different observation units; overlapping windows from S023 are correlated. Activity differences in RMS are descriptive for this recording, not classification performance or a finding about independent participants.

**Limits:** No filtering, fitted scaling, participant-level splitting, model fitting, calibration or independent Test scoring is carried out here. The initial examination of S023 means the later S023/S032 qualified evaluation cannot be described as fully blind or externally validated.

**Next dependency:** The multi-participant Development stage will construct its own appropriately grouped cohort or consume established study-level output files; it must not mistake this single-participant demonstration for the 29-person Development dataset. The next-stage data producer should save its current multi-person arrays and metadata once so later modelling stages can reuse those files rather than repeat windowing and feature engineering.